# 04 — Audit Pass B: Confounding Census

**Run this before notebook 05.**  
Pass B characterizes cohort imbalance and confounding BEFORE sample-size counts are examined.

**Prerequisites:** notebooks 01–03 complete (manifest has ALT labels and SV architecture).

**Decision gate:** if any single cohort contributes >70% of either class in the primary contrast,  
halt and agree on a stratification strategy before running notebook 05.

**Output:** `data/processed/audit_passB_report.md`

In [4]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

# Works whether JupyterLab was launched from the project root OR from notebooks/
_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import log_versions
import pandas as pd

vers = log_versions()
print(f'Run: {RUN_TIMESTAMP}')
print(f'ROOT: {ROOT}')

Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  pyranges: 0.1.4
  openpyxl: 3.1.5
  requests: 2.33.1
  GEOparse: 2.0.4
Run: 2026-05-30T21:04:27Z
ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline


## Load manifest

In [5]:
MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH)

for col in manifest.columns:
    if col.startswith('include_'):
        manifest[col] = manifest[col].astype(str).str.lower().map(
            {'true': True, 'false': False, '1': True, '0': False}
        ).fillna(False)

print(f'Manifest rows: {len(manifest)}')
print(manifest['cohort'].value_counts().to_string())

Manifest rows: 115
cohort
DepMap_26Q1           35
PCGP_2014             34
de_Nonneville_2022    22
Lawlor_2026_GCCRI     17
Taglialatela_2026      7


## Run Pass B: all contingency tables

In [6]:
from src.audit import run_pass_b, contingency_to_markdown

tables, warnings = run_pass_b(manifest)

print(f'Tables generated: {len(tables)}')
print(f'Warnings:         {len(warnings)}')
for w in warnings:
    print(f'\n⚠️  {w}')


Tables generated: 27
Warnings:         15

⚠️  COHORT DOMINANCE: [STRICT] cohort × tcab1_genomic_status — cohort 'PCGP_2014' contributes 100% of class 'intact' (threshold: 70%). Do not proceed to Pass A until stratification strategy is agreed.

⚠️  COHORT DOMINANCE: [STRICT] cohort × alt_status (provenance — Aim 1 universe) — cohort 'PCGP_2014' contributes 73% of class 'alt_gold_positive' (threshold: 70%). Do not proceed to Pass A until stratification strategy is agreed.

⚠️  COHORT DOMINANCE: [STRICT] cohort × alt_status (provenance — Aim 1 universe) — cohort 'PCGP_2014' contributes 100% of class 'alt_negative_high_conf' (threshold: 70%). Do not proceed to Pass A until stratification strategy is agreed.

⚠️  INFO: [STRICT] cohort × alt_status (provenance — Aim 1 universe) — cohort 'DepMap_26Q1' contributes 100% of class 'alt_silver' (label-source coverage, not arm imbalance).

⚠️  INFO: [STRICT] cohort × alt_status (provenance — Aim 1 universe) — cohort 'PCGP_2014' contributes 100% of

## Display tables inline

In [7]:
from IPython.display import display, Markdown

for t in tables:
    md = contingency_to_markdown(
        t['counts'], t['row_pct'], t['chi2'], t['p_val'], t['title']
    )
    display(Markdown(md))

### [STRICT] cohort × tcab1_genomic_status

**Counts:**

| cohort            |   disrupted_high_conf |   intact |
|:------------------|----------------------:|---------:|
| DepMap_26Q1       |                     2 |        0 |
| Lawlor_2026_GCCRI |                     4 |        0 |
| PCGP_2014         |                     2 |       32 |


**Row %:**

| cohort            |   disrupted_high_conf |   intact |
|:------------------|----------------------:|---------:|
| DepMap_26Q1       |                 100   |      0   |
| Lawlor_2026_GCCRI |                 100   |      0   |
| PCGP_2014         |                   5.9 |     94.1 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [STRICT] cohort × alt_status (provenance — Aim 1 universe)

**Counts:**

| cohort            |   alt_gold_positive |   alt_negative_high_conf |   alt_silver |   alt_unknown |
|:------------------|--------------------:|-------------------------:|-------------:|--------------:|
| DepMap_26Q1       |                   0 |                        0 |            2 |             0 |
| Lawlor_2026_GCCRI |                   4 |                        0 |            0 |             0 |
| PCGP_2014         |                  11 |                        2 |            0 |            21 |


**Row %:**

| cohort            |   alt_gold_positive |   alt_negative_high_conf |   alt_silver |   alt_unknown |
|:------------------|--------------------:|-------------------------:|-------------:|--------------:|
| DepMap_26Q1       |                 0   |                      0   |          100 |           0   |
| Lawlor_2026_GCCRI |               100   |                      0   |            0 |           0   |
| PCGP_2014         |                32.4 |                      5.9 |            0 |          61.8 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [STRICT] tcab1_genomic_status × tp53_status

**Counts:**

| tcab1_genomic_status   |   altered |   not_altered |   unknown |
|:-----------------------|----------:|--------------:|----------:|
| disrupted_high_conf    |         3 |             5 |         0 |
| intact                 |         9 |             0 |        23 |


**Row %:**

| tcab1_genomic_status   |   altered |   not_altered |   unknown |
|:-----------------------|----------:|--------------:|----------:|
| disrupted_high_conf    |      37.5 |          62.5 |       0   |
| intact                 |      28.1 |           0   |      71.9 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [STRICT] tcab1_genomic_status × alt_status

**Counts:**

| tcab1_genomic_status   |   alt_gold_positive |   alt_negative_high_conf |   alt_silver |   alt_unknown |
|:-----------------------|--------------------:|-------------------------:|-------------:|--------------:|
| disrupted_high_conf    |                   6 |                        0 |            2 |             0 |
| intact                 |                   9 |                        2 |            0 |            21 |


**Row %:**

| tcab1_genomic_status   |   alt_gold_positive |   alt_negative_high_conf |   alt_silver |   alt_unknown |
|:-----------------------|--------------------:|-------------------------:|-------------:|--------------:|
| disrupted_high_conf    |                75   |                      0   |           25 |           0   |
| intact                 |                28.1 |                      6.2 |            0 |          65.6 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [STRICT] tcab1_genomic_status × telomerase_status

**Counts:**

| tcab1_genomic_status   |   indeterminate |   unknown |
|:-----------------------|----------------:|----------:|
| disrupted_high_conf    |               2 |         6 |
| intact                 |               0 |        32 |


**Row %:**

| tcab1_genomic_status   |   indeterminate |   unknown |
|:-----------------------|----------------:|----------:|
| disrupted_high_conf    |              25 |        75 |
| intact                 |               0 |       100 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [STRICT] cohort × sample_type

**Counts:**

| cohort            |   cell_line |   pdx |   tumor |
|:------------------|------------:|------:|--------:|
| DepMap_26Q1       |           2 |     0 |       0 |
| Lawlor_2026_GCCRI |           0 |     4 |       0 |
| PCGP_2014         |           0 |     0 |      34 |


**Row %:**

| cohort            |   cell_line |   pdx |   tumor |
|:------------------|------------:|------:|--------:|
| DepMap_26Q1       |         100 |     0 |       0 |
| Lawlor_2026_GCCRI |           0 |   100 |       0 |
| PCGP_2014         |           0 |     0 |     100 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

... (17 outputs left out)

### [BROAD] source_confidence × alt_status

**Counts:**

| source_confidence   |   alt_gold_positive |   alt_negative_high_conf |   alt_silver |   alt_unknown |
|:--------------------|--------------------:|-------------------------:|-------------:|--------------:|
| computational       |                   0 |                        0 |            4 |             0 |
| peer_reviewed       |                  19 |                        7 |            0 |            21 |
| preprint            |                   4 |                        0 |            0 |             0 |
| unknown             |                   0 |                        0 |            0 |             2 |


**Row %:**

| source_confidence   |   alt_gold_positive |   alt_negative_high_conf |   alt_silver |   alt_unknown |
|:--------------------|--------------------:|-------------------------:|-------------:|--------------:|
| computational       |                 0   |                      0   |          100 |           0   |
| peer_reviewed       |                40.4 |                     14.9 |            0 |          44.7 |
| preprint            |               100   |                      0   |            0 |           0   |
| unknown             |                 0   |                      0   |            0 |         100   |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [STRICT] cohort × alt_call_direction (Aim 2 arm balance — include_aim2_strict)

**Counts:**

| cohort             |   negative |   positive |
|:-------------------|-----------:|-----------:|
| DepMap_26Q1        |          8 |         12 |
| Lawlor_2026_GCCRI  |          1 |         16 |
| PCGP_2014          |          2 |         11 |
| Taglialatela_2026  |          2 |          5 |
| de_Nonneville_2022 |          6 |         16 |


**Row %:**

| cohort             |   negative |   positive |
|:-------------------|-----------:|-----------:|
| DepMap_26Q1        |       40   |       60   |
| Lawlor_2026_GCCRI  |        5.9 |       94.1 |
| PCGP_2014          |       15.4 |       84.6 |
| Taglialatela_2026  |       28.6 |       71.4 |
| de_Nonneville_2022 |       27.3 |       72.7 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [PRAGMATIC] cohort × alt_call_direction (Aim 2 arm balance — include_aim2_pragmatic)

**Counts:**

| cohort             |   negative |   positive |
|:-------------------|-----------:|-----------:|
| DepMap_26Q1        |         13 |         18 |
| Lawlor_2026_GCCRI  |          1 |         16 |
| PCGP_2014          |          2 |         11 |
| Taglialatela_2026  |          2 |          5 |
| de_Nonneville_2022 |          6 |         16 |


**Row %:**

| cohort             |   negative |   positive |
|:-------------------|-----------:|-----------:|
| DepMap_26Q1        |       41.9 |       58.1 |
| Lawlor_2026_GCCRI  |        5.9 |       94.1 |
| PCGP_2014          |       15.4 |       84.6 |
| Taglialatela_2026  |       28.6 |       71.4 |
| de_Nonneville_2022 |       27.3 |       72.7 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

### [BROAD] cohort × alt_call_direction (Aim 2 arm balance — include_aim2_broad)

**Counts:**

| cohort             |   negative |   positive |
|:-------------------|-----------:|-----------:|
| DepMap_26Q1        |         13 |         18 |
| Lawlor_2026_GCCRI  |          1 |         16 |
| PCGP_2014          |          2 |         11 |
| Taglialatela_2026  |          2 |          5 |
| de_Nonneville_2022 |          6 |         16 |


**Row %:**

| cohort             |   negative |   positive |
|:-------------------|-----------:|-----------:|
| DepMap_26Q1        |       41.9 |       58.1 |
| Lawlor_2026_GCCRI  |        5.9 |       94.1 |
| PCGP_2014          |       15.4 |       84.6 |
| Taglialatela_2026  |       28.6 |       71.4 |
| de_Nonneville_2022 |       27.3 |       72.7 |


*(chi-squared not computed — expected counts < 5 or insufficient categories)*

## Decision gate

If any warning fires above, do NOT run notebook 05 until the cohort imbalance  
is documented and a stratification strategy is agreed upon.  
Set `HALT_OVERRIDE = True` (and document the strategy below) to proceed anyway.

In [11]:
HALT_OVERRIDE = True
STRATIFICATION_NOTES = """
Pass B warnings (direction-aware re-run, 2026-05-30, kernel-restart-clean):

(1) tcab1_genomic_status warnings: PCGP is the only cohort with architecture-
    confirmed intact OS samples; DepMap is the only cohort without architecture
    data. These are structural data-availability findings, not within-cohort
    confounders that stratification could resolve. Documented in
    docs/feasibility_decision_memo.md.

(2) alt_status warnings on `(provenance — Aim 1 universe)` tables: these
    describe label-source coverage (PCGP carries most peer-reviewed tumor
    labels; Lawlor's preprint contributes all PDX labels; DepMap holds
    silver labels). They are provenance facts, not arm-imbalance issues for
    Aim 2 modeling. Aim 2 arm balance is reported separately on the new
    cohort × alt_call_direction tables.

(3) Cohort × alt_call_direction (the modeling-relevant table for Aim 2):
    NO cohort-dominance warnings under any definition. DepMap is 8 neg /
    12 pos under strict (40/60 split — best-balanced cohort). Aim 2 modeling
    universe is internally clean.

(4) Aim 2 modeling primary will restrict to DepMap_26Q1 only (single
    platform, single sample type — cell line) to eliminate any residual
    cross-cohort heterogeneity. Within-cohort cohort × direction is by
    construction one-cohort. The 79/90-sample cross-cohort universes serve
    only as the audit's documented "potential" universe; the analytical
    universe is DepMap-only.

(5) Aim 1 remains MARGINAL primary, GO TP53-restricted, NOT ESTIMABLE under
    Stage 0 RNA-backing diagnostic (notebook 06). Lawlor PDX RNA request
    (Route C) is the active recovery path.

Override approved 2026-05-30 with these caveats documented in
docs/feasibility_decision_memo.md and label_spec_v1.0.md.
"""

## Write Pass B report

This cell generates the report. `tables` and `warnings` must exist from the cell above —  
if you see a `NameError`, run the **Run Pass B** cell first.

In [12]:
from src.audit import write_pass_b_report

REPORT_PATH = ROOT / 'data' / 'processed' / 'audit_passB_report.md'
write_pass_b_report(tables, warnings, REPORT_PATH)
print(f'Report written: {REPORT_PATH}')


Pass B report written to /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/processed/audit_passB_report.md
Report written: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/processed/audit_passB_report.md


## Notebook summary

In [13]:
report_lines = [
    '# Notebook 04 — Pass B Report',
    f'Run: {RUN_TIMESTAMP}',
    '',
    f'Tables generated: {len(tables)}',
    f'Warnings:         {len(warnings)}',
    f'Halt override:    {HALT_OVERRIDE}',
    '',
    '## Warnings',
    *([f'- {w}' for w in warnings] if warnings else ['- None']),
    '',
    f'Full report: data/processed/audit_passB_report.md',
]
nb_report = ROOT / 'data' / 'processed' / 'notebook_04_report.md'
nb_report.write_text('\n'.join(report_lines))
print('\n'.join(report_lines))

# Notebook 04 — Pass B Report
Run: 2026-05-30T21:04:27Z

Tables generated: 27
Warnings:         15
Halt override:    True

## Warnings
- COHORT DOMINANCE: [STRICT] cohort × tcab1_genomic_status — cohort 'PCGP_2014' contributes 100% of class 'intact' (threshold: 70%). Do not proceed to Pass A until stratification strategy is agreed.
- COHORT DOMINANCE: [STRICT] cohort × alt_status (provenance — Aim 1 universe) — cohort 'PCGP_2014' contributes 73% of class 'alt_gold_positive' (threshold: 70%). Do not proceed to Pass A until stratification strategy is agreed.
- COHORT DOMINANCE: [STRICT] cohort × alt_status (provenance — Aim 1 universe) — cohort 'PCGP_2014' contributes 100% of class 'alt_negative_high_conf' (threshold: 70%). Do not proceed to Pass A until stratification strategy is agreed.
- INFO: [STRICT] cohort × alt_status (provenance — Aim 1 universe) — cohort 'DepMap_26Q1' contributes 100% of class 'alt_silver' (label-source coverage, not arm imbalance).
- INFO: [STRICT] cohort × alt